# Reproduce the main results

Self-contained: every cell can be run top to bottom on a fresh kernel (`uv run jupyter lab`).
It re-trains the four from-scratch classifiers with the hyper-parameters selected on the **validation** split
(`results/tuning/*.json`), evaluates **train / validation / test**, and checks the numbers against the
committed run in `results/final/`. Set `QUICK = True` for a 2-minute sanity run on a subset.

Protocol: stratified 50 000 / 10 000 train/validation split of the official training set (seed 42), official 10 000 test images,
normalisation statistics from the training part only. The test split is used here only for reporting.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments.final import train_final
from src.experiments.search import CLASS_NAME, SPECS
from src.preprocessing.preprocess import load_dataset

QUICK = False   # True -> train on 3 000 images only (does not reproduce the reported numbers)
OUT = ROOT / "results" / ("_notebook_quick" if QUICK else "_notebook_run")

## 1. Data and split

In [ ]:
data, meta = load_dataset()
if QUICK:
    data = {k: (v[:3000] if k == "X_train" or k == "y_train" else v) for k, v in data.items()}
print(json.dumps(meta, indent=2))
print({k: v.shape for k, v in data.items()})
print("class counts (train):", np.bincount(data["y_train"]))
committed = json.loads((ROOT / "results" / "final" / "GaussianNB.json").read_text())
assert committed["dataset"]["split_fingerprint"] == meta["split_fingerprint"], "different split than the committed results"

## 2. Selected hyper-parameters (chosen on validation only)

In [ ]:
best = {m: json.loads((ROOT / "results" / "tuning" / f"{CLASS_NAME[m]}.json").read_text())["best_params"] for m in SPECS}
pd.Series({CLASS_NAME[m]: p for m, p in best.items()}).to_frame("best_params")

## 3. Train and evaluate the four models

In [ ]:
results = {}
for m in SPECS:
    _, res = train_final(m, best[m], data, meta, tracker=None, save=False, out_dir=OUT, repeats=2)
    results[m] = res

## 4. Benchmark table (train / validation / test)

In [ ]:
rows = []
for m, r in results.items():
    s = r["splits"]
    rows.append({"model": CLASS_NAME[m],
                 "acc_train": s["train"]["accuracy"], "acc_val": s["val"]["accuracy"], "acc_test": s["test"]["accuracy"],
                 "precision_macro_test": s["test"]["precision_macro"], "recall_macro_test": s["test"]["recall_macro"],
                 "f1_macro_test": s["test"]["f1_macro"], "fit_s": r["train_time_sec"],
                 "predict_10k_s": s["test"]["predict_time_sec"]})
table = pd.DataFrame(rows).set_index("model")
table.style.format("{:.4f}")

## 5. Confusion matrices (test)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, (m, r) in zip(axes, results.items()):
    cm = np.array(r["splits"]["test"]["confusion_matrix"])
    ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(10):
        for j in range(10):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] / cm[i].sum() > .55 else "#333")
    ax.set_title(f"{CLASS_NAME[m]}  acc={r['splits']['test']['accuracy']:.4f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

## 6. Agreement with the committed run (`results/final`)
Runs are seeded, so accuracies should match exactly on the same machine/library versions; small differences are possible
across hardware/BLAS builds (floating-point summation order).

In [ ]:
if QUICK:
    print("QUICK mode: skipped (different training set)")
else:
    cmp = []
    for m, r in results.items():
        ref = json.loads((ROOT / "results" / "final" / f"{CLASS_NAME[m]}.json").read_text())
        cmp.append({"model": CLASS_NAME[m],
                    "test_acc_now": r["splits"]["test"]["accuracy"],
                    "test_acc_committed": ref["splits"]["test"]["accuracy"],
                    "abs_diff": abs(r["splits"]["test"]["accuracy"] - ref["splits"]["test"]["accuracy"])})
    cmp = pd.DataFrame(cmp).set_index("model"); display(cmp.style.format("{:.5f}"))
    assert (cmp["abs_diff"] < 0.002).all(), "results drifted from the committed run"